---
title: "Alignment Summary - `r params$sample`"
include-in-header:
  - text: |
      <style>
      .BioCircosHISTOGRAMTooltip {
        z-index: 10;
      }
      </style>
jupyter: ir
params:
  bxstats: 'sample.bxstats.gz'
  coverage: 'sample.cov.gz'
  molcov: 'sample.molcov.gz'
  mol_dist: 75000
  windowsize: 50000
  contigs: "default"
  sample: "sample"
---

`r format(Sys.time(), '🗓️ %d %B, %Y 🕔 %H:%M')`

# Barcode Stats

In [ ]:
library(dplyr)
library(highcharter)
library(DT)
library(BioCircos)

In [ ]:
infile <- params$bxstats
samplename <- params$sample
sm_moldist <- params$mol_dist
bamfile <- paste0(samplename, ".bam")
tb <- read.table(infile, header = T, sep = "\t") %>% select(-start, -end)
if(nrow(tb) == 0){
  print(paste("Input data file",infile, "is empty"))
  knitr::knit_exit()
}
tb$valid <- tb$molecule != "invalidBX"

In [ ]:
valids <- tb[tb$valid, -ncol(tb)]
if(nrow(valids) == 0){
  VALID_PRESENT <- FALSE
  SKIPTEXT <- TRUE
} else {
  VALID_PRESENT <- TRUE
  SKIPTEXT <- FALSE
}

In [ ]:
invalids <- tb[!(tb$valid), -ncol(tb)]
valids$molecule <- as.integer(valids$molecule)
nBX <- group_by(valids, contig) %>% 
  summarize(nBX = length(molecule))

avgBX <- round(mean(nBX$nBX), digits = 2)
totuniqBX <- read.table(infile, header = F, sep = "\n", as.is = T, skip = nrow(tb) + 1, comment.char = "+")
totuniqBX <- gsub("#total unique barcodes: ", "", totuniqBX) |> as.integer()

tot_valid <- sum(valids$reads)
tot_invalid <- sum(invalids$reads)
n_reads <- tot_valid + tot_invalid
perc_valid <- round(tot_valid / n_reads * 100, 2)

non_singletons <- valids[valids$reads >= 2, ]
n_non_singleton_mol <- nrow(non_singletons)

In [ ]:
NX <- function(lengths, X=50){
  lengths <- as.numeric(sort(lengths, decreasing=TRUE))
  index <- which(cumsum(lengths) / sum(lengths) >= X/100)[1]
  return(lengths[index])
}

## 

In [ ]:
list(
  color = "light",
  value = prettyNum(length(unique(tb$contig)), big.mark = ",")
)

In [ ]:
list(
  color = "#d4d4d4",
  value = prettyNum(totuniqBX, big.mark = ",")
)

In [ ]:
if(sm_moldist > 0){
  boxval <- paste0(sm_moldist/1000, "kb")
} else {
  boxval <- "NA"
}
list(
  color = "#d4d4d4",
  value = boxval
)

In [ ]:
list(
  color = "#d4d4d4",
  value = prettyNum(length(unique(valids$molecule)), big.mark = ",")
)

In [ ]:
list(
  color = "#d4d4d4",
  value = paste0(perc_valid, "%")
)

In [ ]:
if (VALID_PRESENT){
  boxval <- paste0(round(100 - sum(valids$reads < 2)/nrow(valids) * 100, 2),"%")
} else {
  boxval <- "NA"
}
list(
  color = "#84bd9e",
  value = boxval
)

In [ ]:
if (VALID_PRESENT){
  boxval <- prettyNum(avgBX, big.mark = ",")
} else {
  boxval <- "NA"
}
list(
  color = "#d4d4d4",
  value = boxval
)

## 

::: {.card fill="false"}

In [ ]:
if(!VALID_PRESENT){
  cat(paste0("**`",infile,"` has no valid barcodes, skipping reporting on linked-read metrics.**"))
} else {
  summary_table <- valids %>% 
      group_by(contig) %>%
      summarize(valid_records = sum(reads), molecules = length(molecule),n50 = NX(length_inferred, 50), n75 = NX(length_inferred, 75), n90 = NX(length_inferred, 90))
  summ_invalid <- invalids %>% group_by(contig) %>% summarize(invalid_records = sum(reads))
  summary_table <- left_join(summary_table, summ_invalid, by = "contig", keep = T) %>% select(1, 2, 8, 3:6)
  summary_table$invalid_records[is.na(summary_table$invalid_records)] <- 0

  column_description <- c(
    "name of the contig",
    "number of valid linked-read barcodes",
    "number of invalid linked-read barcodes (dependent on linked-read type)",
    "the unique DNA molecules as inferred from linked-read barcodes",
    "N50 of inferred molecules",
    "N75 of inferred molecules",
    "N90 of inferred molecules"
    )

  headerCallback <- c(
    "function(thead, data, start, end, display){",
    paste0(" var tooltips = ['", paste(column_description, collapse = "','"), "'];"),
    "  for(var i=0; i<tooltips.length; i++){",
    "    $('th:eq('+i+')',thead).attr('title', tooltips[i]);",
    "  }",
    "}"
  )

  DT::datatable(
    summary_table,
    rownames = F,
    colnames = c("contig", "valid BX records", "invalid BX records", "molecules", "N50", "N75", "N90"),
    fillContainer = T,
    options = list(
      dom = 'Brtp',
      paging = TRUE,
      buttons = list(list(extend = "csv",filename = paste0(samplename ,"_molecule_NX"))),
      headerCallback = JS(headerCallback)
    )
  )
}

:::

## Reads per molecule

::: {.card title="Reads per Molecule"}
This chart shows the distribution of the number of reads per linked molecule. That is, how many alignments are associated with a unique molecule. This excludes the number of reads associated with invalid or absent linked-read barcodes. There are two distributions: one with `all` the alignments (orange) and another with only `linked reads` (green). The percent of molecules in the `linked reads` distribution is relative to the total number of non-singleton molecules.

In [ ]:
hs <- hist(valids$reads, breaks = min(valids$reads):max(valids$reads), plot = F)
hs_mult <- hist(non_singletons$reads, breaks = hs$breaks, plot = F)
hs$counts <- round(hs$counts / sum(hs$counts) * 100, 4)
hs_mult$counts <- round(hs_mult$counts / sum(hs_mult$counts) * 100, 4)
hs <- data.frame(val = hs$breaks[-1], freq = hs$counts)
hs_mult <- data.frame(val = hs_mult$breaks[-1], freq = hs_mult$counts)

highchart() |>
  hc_chart(type = "area", animation = F) |>
  hc_add_series(data = hs, type = "areaspline", hcaes(x = val, y = freq), color = "#eeaf61", name = "all", marker = list(enabled = FALSE)) |>
  hc_add_series(data = hs_mult, type = "areaspline", hcaes(x = val, y = freq), color = "#84bd9e", name = "linked reads", marker = list(enabled = FALSE)) |>
  hc_title(text = "Reads Per Molecule") |>
  hc_xAxis(title = list(text = "reads per molecule")) |>
  hc_yAxis(max = 100, title = list(text = "% molecules")) |>
  hc_caption(
    text = paste0("unique molecules: ", totuniqBX, " (", n_non_singleton_mol, " linked reads)")
  ) |>
  hc_tooltip(crosshairs = TRUE) |>
  hc_exporting(enabled = T, filename = paste0(samplename, ".readsper"),
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  )

:::

### bases per desc

::: {.card title="Total Bases Aligned"}
This is a frequency distribution showing the number of base pairs aligned per unique molecule. These data are shown in 500 bp bins. Be aware that the Y axis may not start at `0`. There are two distributions: one with `all` the alignments (orange) and another with only `linked reads` (green). The percent of molecules in the `linked reads` distribution is relative to the total number of non-singleton molecules.

In [ ]:
hs <- hist(round(valids$aligned_bp, -2), breaks = 50,  plot = F)
hs_mult <- hist(round(non_singletons$aligned_bp, -2), breaks = hs$breaks,  plot = F)
hs$counts <- round(hs$counts / sum(hs$counts)*100,4)
hs_mult$counts <- round(hs_mult$counts / sum(hs_mult$counts)*100,4)
hs <- data.frame(val = hs$breaks[-1], freq = hs$counts)
hs_mult <- data.frame(val = hs_mult$breaks[-1], freq = hs_mult$counts)

highchart() |>
  hc_chart(type = "area", animation = F) |>
  hc_add_series(data = hs, type = "areaspline", hcaes(x = val, y = freq), color = "#eeaf61", name = "all", marker = list(enabled = FALSE)) |>
  hc_add_series(data = hs_mult, type = "areaspline", hcaes(x = val, y = freq), color = "#84bd9e", name = "linked reads", marker = list(enabled = FALSE)) |>
  hc_title(text = "Bases Aligned Per Molecule") |>
  hc_xAxis(title = list(text = "aligned bases per molecule")) |>
  hc_yAxis(max = 100, title = list(text = "% molecules")) |>
  hc_caption(
    text = paste0("unique molecules: ", totuniqBX, " (", n_non_singleton_mol, " linked reads)")
  ) |>
  hc_tooltip(crosshairs = TRUE) |>
  hc_exporting(enabled = T, filename = paste0(samplename, ".basesper"),
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  )

:::

## 

::: {.card title="Inferred Molecule Lengths"}
This is the frequency distribution of molecule lengths inferred from the first and last alignment positions along a contig for all alignments associated with a single linked-read barcode on a given contig. Be aware that the Y axis may not start at `0`. There are two distributions: one with `all` the alignments (orange) and another with only `linked reads` (green). The percent of molecules in the `linked reads` distribution is relative to the total number of non-singleton molecules.

In [ ]:
hs <- hist(round(valids$length_inferred / 1000,0),breaks = 25,plot = F)
hs_mult <- hist(round(non_singletons$length_inferred / 1000,0),breaks = hs$breaks,plot = F)
hs$counts <- round(hs$counts / sum(hs$counts)*100,2)
hs_mult$counts <- round(hs_mult$counts / sum(hs_mult$counts)*100,2)
hs <- data.frame(val = hs$breaks[-1], freq = hs$counts)
hs_mult <- data.frame(val = hs_mult$breaks[-1], freq = hs_mult$counts)

highchart() |>
  hc_chart(type = "area", animation = F) |>
  hc_add_series(data = hs, type = "areaspline", hcaes(x = val, y = freq), color = "#eeaf61", name = "all", marker = list(enabled = FALSE)) |>
  hc_add_series(data = hs_mult, type = "areaspline", hcaes(x = val, y = freq), color = "#84bd9e", name = "linked reads", marker = list(enabled = FALSE)) |>
  hc_title(text = "Inferred Molecule Length") |>
  hc_subtitle(text = "lengths reported as kilobases (kbp)") |>
  hc_xAxis(title = list(text = "Inferred Molecule length (kbp)"), type = "logarithmic") |>
  hc_yAxis(max = 100, title = list(text = "% of molecules")) |>
  hc_caption(
    text = paste0("unique molecules: ", totuniqBX, " (", n_non_singleton_mol, " linked reads)")
  ) |>
  hc_tooltip(crosshairs = TRUE) |>
  hc_exporting(enabled = T, filename = paste0(samplename, ".mollen"),
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  )

:::

## 

::: {.card title="Molecule Read Coverage"}
This chart shows molecule coverage breadth as computed by the number of bases aligned to that molecule. In other words, "*how much of each unique long molecule is actually sequenced/aligned?*" Keep in mind there is a distinction between sequences and alignments, since some sequences belonging to a particular molecule may not align well and wouldn't appear in the alignment data.

In [ ]:
hs <- hist(round(valids$coverage_bp, 0), breaks = seq(0, 1, by = 0.05), plot = F)
hs_mult <- hist(round(non_singletons$coverage_bp, 0), breaks = hs$breaks, plot = F)
hs$counts <- round(hs$counts / sum(hs$counts)*100, 4)
hs_mult$counts <- round(hs_mult$counts / sum(hs_mult$counts)*100, 4)
hs <- data.frame(val = hs$breaks[-1], freq = hs$counts)
hs_mult <- data.frame(val = hs_mult$breaks[-1], freq = hs_mult$counts)

highchart() |>
  hc_chart(type = "area", animation = F) |>
  hc_add_series(data = hs, type = "areaspline", hcaes(x = val, y = freq), color = "#eeaf61", name = "all", marker = list(enabled = FALSE)) |>
  hc_add_series(data = hs_mult, type = "areaspline", hcaes(x = val, y = freq), color = "#84bd9e", name = "linked reads", marker = list(enabled = FALSE)) |>
  hc_title(text = "Percent Molecule Coverage Breadth by Aligned Bases") |>
  hc_xAxis(title = list(text = "% molecule covered")) |>
  hc_yAxis(max = 100, title = list(text = "% of molecules")) |>
  hc_caption(
    text = paste0("unique molecules: ", totuniqBX, " (", n_non_singleton_mol, " linked reads)")
  ) |>
  hc_tooltip(crosshairs = TRUE) |>
  hc_exporting(enabled = T, filename = paste0(samplename, ".bp.molcov"),
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  )

:::

::: {.card title="Molecular Inferred Coverage"}
This chart shows *molecule coverage breadth* as computed by the *inferred fragment* from which the aligned sequences originate. Unlike read coverage, this calculation takes into account the total insert size, including unsequenced gaps between the forward and reverse reads.

In [ ]:
hs <- hist(round(valids$coverage_inserts, 0), breaks = seq(0, 1, by = 0.05), plot = F)
hs_mult <- hist(round(non_singletons$coverage_inserts, 0), breaks = hs$breaks, plot = F)
hs$counts <- round(hs$counts / sum(hs$counts)*100,2)
hs_mult$counts <- round(hs_mult$counts / sum(hs_mult$counts)*100,2)
hs <- data.frame(val = hs$breaks[-1], freq = hs$counts)
hs_mult <- data.frame(val = hs_mult$breaks[-1], freq = hs_mult$counts)

highchart() |>
  hc_chart(type = "area", animation = F) |>
  hc_add_series(data = hs, type = "areaspline", hcaes(x = val, y = freq), color = "#eeaf61", name = "all", marker = list(enabled = FALSE)) |>
  hc_add_series(data = hs_mult, type = "areaspline", hcaes(x = val, y = freq), color = "#84bd9e", name = "linked reads", marker = list(enabled = FALSE)) |>
  hc_title(text = "Percent Molecule Coverage Breadth by Inferred Insert Size") |>
  hc_xAxis(title = list(text = "% molecule covered")) |>
  hc_yAxis(max = 100, title = list(text = "% of molecules")) |>
  hc_caption(
    text = paste0("unique molecules: ", totuniqBX, " (", n_non_singleton_mol, " linked reads)")
  ) |>
  hc_tooltip(crosshairs = TRUE) |>
  hc_exporting(enabled = T, filename = paste0(samplename, ".inferred.molcov"),
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  )

:::

# Depth Stats

In [ ]:
# clear out objects in memory to reduce RAM usage
rm(hs)
rm(hs_mult)
rm(valids)
rm(invalids)
rm(non_singletons)

In [ ]:
covfile <- params$coverage
molcovfile <- params$molcov
windowsize <- as.integer(params$windowsize)
samplename <- params$sample

tb <- read.table(covfile, header = F)
if(nrow(tb) == 0){
  print(paste("Input data file", covfile, "is empty"))
  knitr::knit_exit()
} else {
  tb <- tb[, c(1,2,3,6)]
}
tbmol <- read.table(molcovfile, header = F)
if(nrow(tbmol) == 0){
  print(paste("Input data file", molcovfile, "is empty"))
  knitr::knit_exit()
}

In [ ]:
colnames(tb) <- c("contig", "position", "position_end", "depth")
colnames(tbmol) <- c("contig", "position", "position_end", "mol_depth")
tb <- merge(tb, tbmol, by = c("contig", "position", "position_end"))
q99 <- quantile(tb$depth, 0.99)
mol_q99 <- quantile(tb$mol_depth, 0.99)
names(q99) <- NULL
names(mol_q99) <- NULL
windowskb <- round(windowsize/1000, 0)
rm(tbmol)

In [ ]:
global_avg <- mean(tb$depth)
global_sd <- sd(tb$depth)
mol_global_avg <- mean(tb$mol_depth)
mol_global_sd <- sd(tb$mol_depth)

tb$outlier <- tb$depth > q99
outliers <- tb[tb$outlier, -6]
nonoutliers <- tb[!(tb$outlier), -6]
outliers[,5] <- round(outliers[,5], 4)

contig_avg <- group_by(tb, contig) %>%
  summarize(average = mean(depth), mol_average = mean(mol_depth), sdv = sd(depth), mol_sdv = sd(mol_depth))
contig_avg <- rbind(data.frame(contig = "global", average = global_avg, sdv = global_sd, mol_average = mol_global_avg, mol_sdv = mol_global_sd), contig_avg) %>% 
  mutate(average = round(average, 2), sdv = round(sdv, 2), mol_average = round(mol_average, 2), mol_sdv = round(mol_sdv, 2))

## 

In [ ]:
list(
  color = "light",
  value = prettyNum(length(unique(tb$contig)), big.mark = ",")
)

In [ ]:
list(
  color = "#d4d4d4",
  value = paste(windowskb, "kb")
)

In [ ]:
list(
  color = "#d4d4d4",
  value = round(global_avg,2)
)

In [ ]:
list(
  color = "#d4d4d4",
  value = round(global_sd,2)
)

In [ ]:
list(
  color = "#84bd9e",
  value = round(mol_global_avg,2)
)

In [ ]:
list(
  color = "#84bd9e",
  value = round(mol_global_sd,2)
)

## Distdesc header

::: {.card title="Alignment Depth Distribution"}
These are the frequencies of interval coverage across all **`r windowskb` kilobase** intervals for all contigs. For visual clarity, 
the distributions are truncated at the 99% quantile, which is **`r q99`** for the alignments and **`r format(mol_q99, scientific = F)
`** for the inferred molecules. If the linked depth seems suspiciously high, then it's possible (or likely) that you have distant
alignments sharing barcodes but don't actually originate from the same DNA molecule and instead share the barcode by chance. These
are referred to by a handful of names like "barcode clashing" and can be remedied by deconvolution methods at the pre-
or post-alignment stages. Additionally, many downstream linked-read softwares have internal deconvolution methods to account for 
this, so it might not be something that needs addressing at this stage.

In [ ]:
hs <- hist(tb$depth[tb$depth <= q99], breaks = 30, plot = F)
hs_mol <- hist(tb$mol_depth[tb$mol_depth <= mol_q99], breaks = 30, plot = F)
hs$counts <- round(hs$counts / sum(hs$counts) * 100, 2)
hs_mol$counts <- round(hs_mol$counts / sum(hs_mol$counts) * 100, 2)
hs <- data.frame(val = hs$breaks[-1], freq = hs$counts)
hs_mol <- data.frame(val = hs_mol$breaks[-1], freq = hs_mol$counts)

highchart() |>
  hc_chart(type = "area", animation = F) |>
  hc_add_series(data = hs, type = "areaspline", color = "#757575", name = "alignments", hcaes(x = val, y = freq),  marker = list(enabled = FALSE)) |>
  hc_add_series(data = hs_mol, type = "areaspline", hcaes(x = val, y = freq), color = "#9c3b94", name = "inferred molecules", marker = list(enabled = FALSE)) |>
  hc_xAxis(max = mol_q99, title = list(text = "depth")) |>
  hc_yAxis(title = list(text = "% intervals"))  |>
  hc_caption(text = "inferred molecules provide \"linked\" coverage") |>
  hc_title(text = "Distribution of Alignment Depths")  |>
  hc_exporting(
    enabled = T, filename = paste0(samplename, ".cov"),
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  )

:::

## Sumheader

::: {.card title="Average Depth"}
This table shows the global and per-contig average depth and standard deviation per `r windowskb`kbp intervals **including** intervals whose depth is flagged an outlier in the data.

In [ ]:
column_description <- c(
  "name of the contig",
  "average alignment depth",
  "standard deviation of alignment depth",
  "average depth including gaps between linked reads",
  "standard deviation of the depth including gaps between linked reads"
  )

headerCallback <- c(
  "function(thead, data, start, end, display){",
  paste0(" var tooltips = ['", paste(column_description, collapse = "','"), "'];"),
  "  for(var i=0; i<tooltips.length; i++){",
  "    $('th:eq('+i+')',thead).attr('title', tooltips[i]);",
  "  }",
  "}"
)

DT::datatable(
  contig_avg, 
  rownames = F, 
  extensions = 'Buttons', 
  colnames = c('Contig', 'Average Depth', 'Standard Deviation', 'Average Linked Depth', 'Standard Deviation Linked'),
  fillContainer=T,
  options = list(
    dom = 'Brtp',
    scrollX = TRUE,
    paging = TRUE,
    buttons = list(list(extend = "csv",filename = paste0(samplename ,"_align_depth_avg"))),
    headerCallback = JS(headerCallback)
  )
)

:::

### 

::: {.card title="Depth Outliers"}
This table shows the `r windowskb`kbp intervals considered outliers, as determined by having depth greater than the 99% quantile (`r q99`) of aligment depths.

In [ ]:
column_description <- c(
  "name of the contig",
  "the genomic interval on the contig",
  "number of alignments at the interval",
  "number of alignments at the interval, including gaps between linked reads"
  )

headerCallback <- c(
  "function(thead, data, start, end, display){",
  paste0(" var tooltips = ['", paste(column_description, collapse = "','"), "'];"),
  "  for(var i=0; i<tooltips.length; i++){",
  "    $('th:eq('+i+')',thead).attr('title', tooltips[i]);",
  "  }",
  "}"
)

DT::datatable(
  outliers, 
  rownames = F, 
  extensions = 'Buttons', 
  colnames = c('Contig', 'Interval', 'Depth', 'Linked Depth'),
  fillContainer=T,
  options = list(
    dom = 'Brtp',
    scrollX = TRUE,
    paging = TRUE,
    buttons = list(list(extend = "csv",filename = paste0(samplename ,"_align_depth_outlier_avg"))),
    headerCallback = JS(headerCallback)
  ),
)

:::

# Depth Plot

## 

::: {.card title="Depth and Coverage, Visualized" fill="false"}
This is a circular vizualization the depth information across up to 30 of the largest contigs (unless specific contigs were provided). For clarity, this visualization truncates coverage at the 99% quantile (`r q99` for alignments and `r mol_q99` for inferred molecules). If you are unfamiliar with this kind of visualization, it's a circular representation of a linear genome. Each arc (segment) is a different contig, from position 0 to the end of the contig, and is labelled by the contig name. The internal (grey) rings are a histogram where each bar represents the alignment depth at a `r windowskb` kilobase genomic interval. These reads are considered to have *proper* alignment in the `r windowskb` kilobase interval, where "proper" refers to a read not marked as a duplicate or flagged with the SAM `UNMAP`, `SECONDARY`, or `QCFAIL` flags. The outer ring (magenta bars), is the number of *molecules*, i.e. the **linked depth**, which is the alignment depth of the molecules inferred from linked-read data, included unsequenced segments between reads sharing the same linked-read barcode. It's common for the linked depth histogram to gradually increase towards the center of a contig (your plot may resemble petals of a flower) due to the likelihood of linked molecules spanning the center of a contig.
:::

## 

###  {width="20%"}

::: {.card title="Navigating the Plot"}
You may hover your cursor over bars to view their positions and depths, pan by clicking and dragging, and zoom using scroll (mouse or touchpad). In case you become unable to scroll up from the plot due to these interactive features, place your cursor over this left column and you will be able to scroll the report instead of zooming on the plot. Try refreshing the browser window if no plot is appearing in the pane. Note: The depth values are rounded to 4 decimal places.
:::

### 

In [ ]:
plot_contigs <- params$contigs
# Find the 30 largest contigs
contigs <- group_by(tb, contig) %>%
  summarize(size = max(position)) %>%
  arrange(desc(size))
  
if (all(plot_contigs == "default")){
  # limit the data to only the 30 largest contigs
  if (nrow(contigs) > 30){
    .contigs <- contigs[1:30, ]
    tb <- filter(tb, contig %in% .contigs$contig)
  } else {
    .contigs <- contigs
  }
} else {
  .contigs <- filter(contigs, contig %in% plot_contigs)
  tb <- filter(tb, contig %in% plot_contigs)
}

In [ ]:
genomeChr <- .contigs$size
names(genomeChr) <- .contigs$contig
genomeChr <- as.list(genomeChr)
inner <- c(0.2, 0.6)
outer <- c(0.61, 0.97)
tracks <- BioCircosTracklist()

# Add one track for each chromosome
for (i in names(genomeChr)){
  # Define histogram/bars to be displayed
  chrcov <- tb[tb$contig == i,]
  #chrcov$depth
  # Add a track with bars on the i-th chromosome
  tracks <- tracks + BioCircosBarTrack(
    paste0("cov_", i),
    chromosome = i, 
    starts = chrcov$position, ends = chrcov$position_end,
    values = round(pmin(chrcov$depth, q99),3),
    color = "#757575",
    minRadius = inner[1] + 0.02,
    maxRadius = inner[2] - 0.02
  )
  tracks <- tracks + BioCircosBarTrack(
    paste0("molcov_", i),
    chromosome = i, 
    starts = chrcov$position, ends = chrcov$position_end,
    values = round(pmin(chrcov$mol_depth, mol_q99),3),
    color = "#9c3b94",
    minRadius = outer[1] + 0.02,
    maxRadius = outer[2] - 0.02
  )
}
# Add background
tracks <- tracks + BioCircosBackgroundTrack(
  "depth_background",
  fillColors = "#f3f3f3",
  borderColors = "#C9C9C9",
  minRadius = inner[1],
  maxRadius = inner[2]
)
tracks <- tracks + BioCircosBackgroundTrack(
  "molcov_background",
  fillColors = "#f3f3f3",
  borderColors = "#C9C9C9",
  minRadius = outer[1],
  maxRadius = outer[2]
  )
BioCircos(
  tracks,
  displayGenomeBorder = F,
  genome = genomeChr,
  chrPad = 0.02,
  genomeTicksDisplay = F,
  BARMouseOverColor = "#1cff42",
  BARMouseOverTooltipsHtml05 = "Depth: ",
  genomeLabelDy = 0,
  width = "100%",
  height = 1000
)

# Supporting Info

## 

::: {.card title="Interpreting the Supporting File"}
Below are the descriptions of the columns in **`r basename(infile)`**, which was created by Harpy using the included `bx_stats` script. The term `molecule` refers to the `MI:i` tag in the alignments, which is a unique molecule ID given to the original fragment alignments sharing a barcode are inferred to have originated from. The inference takes into account an [alignment distance threshold](https://pdimens.github.io/harpy/haplotagdata/#barcode-thresholds) and that the sequences aligned to the same contig.

contig

:   name of the contig the molecule occurs on

molecule

:   the molecule name as given by the MI:i: tag

reads

:   number of alignments associated with this molecule

start

:   the start position of the first alignment for that molecule

end

:   the end position of the last alignment for that molecule

length_inferred

:   inferred length of the molecule based on the start/end of the alignments sharing the same barcode

percent_coverage

:   what percent of the molecule is represented by sequence alignments

aligned_bp

:   total number of base pairs aligned for that molecule
:::

### 

::: {.card title="Interpreting Linked-Read Terminology"}
BX barcode validity is classified into one of three categories:

Valid

:   A complete BX barcode was present in the read (i.e. no 00 for any segments)

Invalid

:   A barcode was present in the read, but it contained `00` in at least one of the barcode segments (haplotagging), `0` as one of the segments (stlfr), or an `N` (tellseq/10x).

Missing

:   There is no barcode in the read. For technical reasons this is usually equivalent to `invalid`

Linked-read data is specific for the definition of a "molecule":

Unique/Inferred Molecules

:   Given linked-read barcode information, the original piece of DNA from which the sequenced fragments are considered to originate from.

Inferred Sequence

:   While somewhat similar to "inferred molecule", the inferred sequence describes the original DNA fragment that was put on the sequencer. If the fragment was longer than the sequencer could fully sequence, e.g. 400bp fragment and the sequencer can only sequence 300bp, then the inferred sequence is 400bp long, even though only 300bp are represented in the sequence data. If the entire fragment was sequenced, then the inferred length and sequence lengths should be identical.

There are several kinds of "coverage" when working with linked-read data:

Aligned Depth/Coverage

:   The standard interpretation of depth comparing the number of aligned base-pairs to the genome or contigs

Molecule Coverage

:   The coverage breadth or depth of sequences onto *unique molecules* (rather than the genome), as inferred from linked-read barcodes

Linked Depth/Coverage

:   The coverage breadth or depth that *includes unsequenced gaps between linked sequences* that are associated with a single unique molecule. For example, if two 300bp paired-end reads share the same barcode and map 2000bp apart, the calculation *includes* the 1400bp between the sequences as if they were present. This is similar to *inferred sequences* described above, except spanning across linked sequences rather than within a paired-end sequence.
:::